In [ ]:
# @title MTL Pipeline: Attention Pooling + LR Scheduler
import os
import zipfile
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from google.colab import files
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm
from datasets import load_from_disk

# --- 1. Configuration ---
CONFIG = {
    'model_name': 'answerdotai/ModernBERT-base',
    'max_len': 1024,
    'batch_size': 32,
    'epochs': 15,
    'learning_rate': 2e-5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,       # NEW: 10% of steps for warmup
    'lambda_child': 1.0,
    'lambda_consistency': 0.5,
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'patience': 5
}

# --- 2. Hierarchy Definitions ---
EVASION_LABELS = [
    'Claims ignorance', 'Clarification', 'Declining to answer',
    'Deflection', 'Dodging', 'General', 'Implicit', 'Partial/half-answer',
    'Explicit'
]
CLARITY_LABELS = ['Ambivalent', 'Clear Non-Reply', 'Clear Reply']

HIERARCHY_MAP = {
    'Claims ignorance': 'Clear Non-Reply', 'Clarification': 'Clear Non-Reply', 'Declining to answer': 'Clear Non-Reply',
    'Deflection': 'Ambivalent', 'Dodging': 'Ambivalent', 'General': 'Ambivalent',
    'Implicit': 'Ambivalent', 'Partial/half-answer': 'Ambivalent',
    'Explicit': 'Clear Reply'
}

# --- 3. Data Processing ---

def handle_dataset_upload():
    zip_name = 'processed_dataset.zip'
    if os.path.exists('train') and os.path.exists('test'): return
    if not os.path.exists(zip_name):
        uploaded = files.upload()
        if zip_name not in uploaded:
             found = [f for f in uploaded.keys() if f.endswith('.zip')]
             if found: zip_name = found[0]
             else: return
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall(".")

class MTLDataset(Dataset):
    def __init__(self, df, tokenizer, split='train'):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = CONFIG['max_len']
        self.split = split
        self.q_col = 'formatted_question' if 'formatted_question' in df.columns else 'interview_question'
        self.a_col = 'interview_answer'

        self.evasion2idx = {l: i for i, l in enumerate(EVASION_LABELS)}
        self.clarity2idx = {l: i for i, l in enumerate(CLARITY_LABELS)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = f"<QUESTION> {row[self.q_col]} <ANSWER> {row[self.a_col]}"

        inputs = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        child_id = -1
        parent_id = -1

        if self.split == 'train':
            ev_label = row['evasion_label']
            if ev_label in self.evasion2idx:
                child_id = self.evasion2idx[ev_label]
                parent_str = HIERARCHY_MAP[ev_label]
                parent_id = self.clarity2idx[parent_str]

        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'child_labels': torch.tensor(child_id, dtype=torch.long),
            'parent_labels': torch.tensor(parent_id, dtype=torch.long)
        }

# --- 4. Models: Attention Pooling & MTL ---

class AttentionPooling(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.Tanh(),
            nn.Linear(in_dim, 1)
        )

    def forward(self, last_hidden_state, attention_mask):
        w = self.attention(last_hidden_state)
        mask = attention_mask.unsqueeze(-1).float()
        w = w.masked_fill(mask == 0, -1e9)
        w = F.softmax(w, dim=1)
        return torch.sum(last_hidden_state * w, dim=1)

class MTL_ModernBERT(nn.Module):
    def __init__(self, num_child_classes, num_parent_classes, evasion_labels, clarity_labels, hierarchy_map):
        super(MTL_ModernBERT, self).__init__()
        self.bert = AutoModel.from_pretrained(CONFIG['model_name'])
        hidden_dim = self.bert.config.hidden_size
        self.pooler = AttentionPooling(hidden_dim)
        self.dropout = nn.Dropout(0.1)

        # Heads
        self.parent_classifier = nn.Linear(hidden_dim, num_parent_classes)
        self.child_classifier = nn.Linear(hidden_dim, num_child_classes)

        # --- DYNAMIC MAPPING FIX ---
        # Create the index mapping list dynamically
        map_indices = []
        for child_label in evasion_labels:
            parent_label_name = hierarchy_map[child_label]
            parent_idx = clarity_labels.index(parent_label_name)
            map_indices.append(parent_idx)

        # Register as a buffer (part of state_dict, but not a trainable parameter)
        # This automatically moves to GPU when you call model.to(device)
        self.register_buffer('hierarchy_indices', torch.tensor(map_indices, dtype=torch.long))

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = self.pooler(outputs.last_hidden_state, attention_mask)
        pooled_output = self.dropout(pooled_output)

        parent_logits = self.parent_classifier(pooled_output)
        child_logits = self.child_classifier(pooled_output)

        return parent_logits, child_logits

# --- 5. Custom Loss & Evaluation ---

def consistency_loss(parent_logits, child_logits, map_indices):
    """
    Enforces that P(Child) <= P(Parent).
    If a child label (e.g., 'Dodging') is predicted with high probability,
    its parent ('Ambivalent') must also have at least that probability.
    """
    parent_probs = F.softmax(parent_logits, dim=1) # (Batch, Num_Parent)
    child_probs = F.softmax(child_logits, dim=1)   # (Batch, Num_Child)

    # Gather the parent probability corresponding to each child index
    # map_indices is shape (Num_Child), we expand it to (Batch, Num_Child)
    expanded_indices = map_indices.unsqueeze(0).expand(parent_probs.size(0), -1)

    # Select specific parent probs
    selected_parent_probs = torch.gather(parent_probs, 1, expanded_indices)

    # Constraint: Child Prob - Parent Prob should be <= 0
    # Violation: Child Prob > Parent Prob (Diff is positive)
    diff = child_probs - selected_parent_probs
    loss = torch.mean(F.relu(diff))

    return loss

def evaluate_test_set_mtl(model, loader, test_df):
    model.eval()
    all_preds_child = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(CONFIG['device'])
            mask = batch['attention_mask'].to(CONFIG['device'])
            _, child_logits = model(input_ids, mask)
            preds = torch.argmax(child_logits, dim=1).cpu().numpy()
            all_preds_child.extend(preds)

    pred_strings = [EVASION_LABELS[p] for p in all_preds_child]
    aligned_preds = []
    aligned_golds = []

    for i, pred in enumerate(pred_strings):
        if i >= len(test_df): break
        row = test_df.iloc[i]
        valid_labels = set()
        for col in ['annotator1', 'annotator2', 'annotator3']:
            if col in row and pd.notna(row[col]):
                valid_labels.add(row[col])
        if valid_labels:
            aligned_preds.append(pred)
            aligned_golds.append(valid_labels)

    if not aligned_golds: return 0.0

    def get_f1(gold, pred, label):
        TP = FP = FN = 0
        for g_set, p_val in zip(gold, pred):
            if p_val == label and label in g_set: TP += 1
            elif p_val == label and label not in g_set: FP += 1
            elif label in g_set and p_val not in g_set: FN += 1
        prec = TP/(TP+FP) if (TP+FP)>0 else 0
        rec = TP/(TP+FN) if (TP+FN)>0 else 0
        return 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0

    f1_scores = []
    print(f"\n{'Class':<25} | {'F1 Score':<10}")
    print("-" * 38)
    for label in EVASION_LABELS:
        score = get_f1(aligned_golds, aligned_preds, label)
        f1_scores.append(score)
        print(f"{label:<25} | {score:.4f}")

    macro_f1 = sum(f1_scores) / len(f1_scores)
    print("-" * 38)
    print(f"Macro F1 Score: {macro_f1:.4f}")
    return macro_f1

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_score, model, path='best_mtl_model.pth'):
        # CASE 1: First Run
        if self.best_score is None:
            self.save_checkpoint(val_score, model, path)
            self.best_score = val_score

        # CASE 2: Improvement
        elif val_score > self.best_score + self.min_delta:
            self.save_checkpoint(val_score, model, path)
            self.best_score = val_score
            self.counter = 0

        # CASE 3: Stagnation
        else:
            self.counter += 1
            print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, val_score, model, path):
        prev = self.best_score if self.best_score is not None else -float('inf')
        print(f'Validation F1 improved ({prev:.4f} --> {val_score:.4f}). Saving model...')
        torch.save(model.state_dict(), path)

# --- 6. Main Execution ---

def main():
    handle_dataset_upload()

    try:
        full_ds = load_from_disk(".")
    except:
        if os.path.exists('train'):
            from datasets import DatasetDict
            full_ds = DatasetDict({'train': load_from_disk("train"), 'test': load_from_disk("test")})
        else:
            print("❌ No data found. Upload processed_dataset.zip")
            return

    train_df = full_ds['train'].to_pandas()
    test_df = full_ds['test'].to_pandas()

    tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
    tokenizer.add_special_tokens({'additional_special_tokens': ['<QUESTION>', '<ANSWER>']})

    train_ds = MTLDataset(train_df, tokenizer, split='train')
    test_ds = MTLDataset(test_df, tokenizer, split='test')

    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False)

    y_child = [y for y in train_df['evasion_label'] if y in EVASION_LABELS]
    w_child = compute_class_weight('balanced', classes=np.unique(EVASION_LABELS), y=y_child)
    w_child_t = torch.tensor(w_child, dtype=torch.float32).to(CONFIG['device'])

    y_parent = [HIERARCHY_MAP[y] for y in y_child]
    w_parent = compute_class_weight('balanced', classes=np.unique(CLARITY_LABELS), y=y_parent)
    w_parent_t = torch.tensor(w_parent, dtype=torch.float32).to(CONFIG['device'])

    model = MTL_ModernBERT(num_child_classes=len(EVASION_LABELS), num_parent_classes=len(CLARITY_LABELS), evasion_labels=EVASION_LABELS, clarity_labels=CLARITY_LABELS, hierarchy_map=HIERARCHY_MAP)
    model.bert.resize_token_embeddings(len(tokenizer))
    model.to(CONFIG['device'])

    optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])

    # --- NEW: SCHEDULER SETUP ---
    total_steps = len(train_loader) * CONFIG['epochs']
    warmup_steps = int(total_steps * CONFIG['warmup_ratio'])
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )
    # ----------------------------

    early_stopper = EarlyStopping(patience=CONFIG['patience'], min_delta=0.001)

    print("Starting MTL Training (with Attention Pooling & Scheduler)...")
    for epoch in range(CONFIG['epochs']):
        model.train()
        total_loss = 0
        progress = tqdm(train_loader, desc=f"Epoch {epoch+1}")

        for batch in progress:
            b_ids = batch['input_ids'].to(CONFIG['device'])
            b_mask = batch['attention_mask'].to(CONFIG['device'])
            b_child = batch['child_labels'].to(CONFIG['device'])
            b_parent = batch['parent_labels'].to(CONFIG['device'])

            optimizer.zero_grad()

            p_logits, c_logits = model(b_ids, b_mask)

            loss_p = F.cross_entropy(p_logits, b_parent, weight=w_parent_t)
            loss_c = F.cross_entropy(c_logits, b_child, weight=w_child_t)
            loss_cons = consistency_loss(p_logits, c_logits, model.hierarchy_indices)

            loss = loss_p + (CONFIG['lambda_child'] * loss_c) + (CONFIG['lambda_consistency'] * loss_cons)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            scheduler.step() # <--- Step Scheduler here

            total_loss += loss.item()

            progress.set_postfix({
                'L_p': f"{loss_p.item():.2f}",
                'L_c': f"{loss_c.item():.2f}",
                'loss': f"{loss.item():.4f}",
                'lr': f"{scheduler.get_last_lr()[0]:.2e}" # Optional: Track LR
            })

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} | Average Loss: {avg_loss:.4f}")

        macro_f1 = evaluate_test_set_mtl(model, test_loader, test_df)

        early_stopper(macro_f1, model)

        if early_stopper.early_stop:
            print("🛑 Early stopping triggered.")
            break

    if os.path.exists('best_mtl_model.pth'):
        model.load_state_dict(torch.load('best_mtl_model.pth'))
        print("Restored best model weights.")

if __name__ == "__main__":
    main()